# Gene to Protein Analysis

This notebook focuses on analyzing individual gene mutations and their effects on protein sequences using the Ensembl VEP API.

In [5]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import os
os.chdir(os.path.dirname(os.path.abspath('.')))

import json
from datetime import datetime

import src.utils as utils
import src.config as config
import src.gprofiler as gp
import src.vep_pipeline as vp
import src.vep_analysis as va
import src.vep_metrics as vm
import src.proteingym as pg 
import src.ensembl_rest as er
import src.biopython as bp

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Get Gene and Variant Information

First, let's create functions to get information about a gene and its variants.

In [15]:
# Create client
client = er.get_ensembl_client()

# Initialize results dictionary
results = {
    "query_info": {
        "gene": "HBB",
        "variant_id": "rs334",
        "timestamp": datetime.now().isoformat()
    },
    "gene_info": {},
    "sequences": {},
    "variant_effects": {},
    "variation_details": {}
}

# Look up gene
gene_info = client.symbol_lookup(species='homo_sapiens', symbol='HBB')
results["gene_info"] = {
    "id": gene_info.get('id'),
    "location": f"{gene_info.get('seq_region_name')}:{gene_info.get('start')}-{gene_info.get('end')}"
}

# Get variant effects for sickle cell mutation
variant_id = 'rs334'  # This is the sickle cell variant
vep_results = er.get_vep(
    ids=variant_id,
    species='homo_sapiens',
)

if isinstance(vep_results, dict):
    for variant_id, variant_results in vep_results.items():
        results["variant_effects"][variant_id] = []
        
        for result in variant_results:
            for tc in result.get('transcript_consequences', []):
                if tc.get('canonical') == 1:  # Get canonical transcript info
                    transcript_id = tc.get('transcript_id')
                    protein_id = tc.get('protein_id')
                    
                    transcript_data = {
                        "transcript_id": transcript_id,
                        "protein_id": protein_id,
                        "uniprot_id": tc.get('swissprot', [''])[0],
                        "gene_sequence": None,
                        "protein_sequence": None,
                        "variant_effect": {
                            "impact": tc.get('impact'),
                            "hgvs_p": tc.get('hgvsp'),
                            "position": f"{tc.get('protein_start')}-{tc.get('protein_end')}",
                            "amino_acid_change": tc.get('amino_acids')
                        }
                    }
                    
                    # Get gene sequence if transcript_id exists
                    if transcript_id:
                        try:
                            gene_seq_response = client.sequence_id(species='homo_sapiens', id=transcript_id)
                            if isinstance(gene_seq_response, dict):
                                transcript_data["gene_sequence"] = gene_seq_response.get('seq', '')
                            else:
                                transcript_data["gene_sequence"] = str(gene_seq_response)
                        except Exception as e:
                            transcript_data["gene_sequence_error"] = str(e)
                    
                    # Get protein sequence if protein_id exists
                    if protein_id:
                        try:
                            protein_seq_response = client.sequence_id(species='homo_sapiens', id=protein_id)
                            if isinstance(protein_seq_response, dict):
                                transcript_data["protein_sequence"] = protein_seq_response.get('seq', '')
                            else:
                                transcript_data["protein_sequence"] = str(protein_seq_response)
                        except Exception as e:
                            transcript_data["protein_sequence_error"] = str(e)
                    
                    results["variant_effects"][variant_id].append(transcript_data)

# Get variation details
variation_info = er.get_variation(
    variant_id=variant_id,
    species='homo_sapiens'
)
if variation_info:
    results["variation_details"] = {
        "maf": variation_info.get('MAF'),
        "clinical_significance": variation_info.get('clinical_significance')
    }

# Save to JSON file in the results directory
output_file = os.path.join('/home/caom/VEP_protein/results', f"hbb_variant_analysis_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults have been saved to: {output_file}")

Getting variant info:   0%|          | 0/1 [00:00<?, ?it/s]


Results have been saved to: /home/caom/VEP_protein/results/hbb_variant_analysis_20250319_131027.json


In [3]:
# Get haplotypes for HBB transcript
transcript_id = "ENST00000335295"  # This is the canonical transcript ID we found earlier

print("Getting haplotypes for HBB transcript...")
haplotype_results = er.transcript_haplotypes_get(
    ids=[transcript_id],
    species='homo_sapiens',
    params={
        'protein': 1,  # Include protein sequences
        'aligned': 1,  # Get aligned sequences
    }
)

# Save haplotype results
output_file = os.path.join('/home/caom/VEP_protein/results', f"hbb_haplotypes_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
with open(output_file, 'w') as f:
    json.dump(haplotype_results, f, indent=2)

print(f"\nHaplotype results have been saved to: {output_file}")

# Print summary of haplotypes found
if transcript_id in haplotype_results:
    haplotype_data = haplotype_results[transcript_id]
    print(f"\nFound {len(haplotype_data.get('haplotypes', []))} haplotypes for {transcript_id}")
    print("\nHaplotype summary:")
    for idx, hap in enumerate(haplotype_data.get('haplotypes', []), 1):
        print(f"Haplotype {idx}:")
        print(f"- Population frequency: {hap.get('frequency', 'N/A')}")
        print(f"- Number of differences: {len(hap.get('diffs', []))}")

Getting haplotypes for HBB transcript...


Getting haplotypes:   0%|          | 0/1 [00:00<?, ?it/s]


Haplotype results have been saved to: /home/caom/VEP_protein/results/hbb_haplotypes_20250320_162043.json

Found 0 haplotypes for ENST00000335295

Haplotype summary:


In [7]:
import src.ensembl_rest as er
import src.haplosaurus as hs
import pandas as pd

def get_gene_sequences(gene_symbols, species="homo_sapiens"):
    """
    Get gene information and their corresponding protein/CDS sequences.
    
    Args:
        gene_symbols (list): List of gene symbols (e.g., ["BRCA1", "BRCA2", "TP53"])
        species (str): Species name (default: "homo_sapiens")
        
    Returns:
        tuple: (gene_data, sequences_dict, missing_seqs)
            - gene_data: DataFrame with gene information
            - sequences_dict: Dictionary with protein sequences
            - missing_seqs: List of transcript IDs with no sequences
    """
    # 1. Get gene info using xref_external first to get Ensembl IDs
    xref_results = er.xref_external(
        ids=gene_symbols,
        species=species,
        verbose=True
    )
    
    # Extract Ensembl gene IDs
    gene_ids = []
    gene_map = {}  # Map to keep track of gene symbol to Ensembl ID mapping
    for gene_symbol, data in xref_results.items():
        if isinstance(data, dict) and 'gene' in data:
            gene_ids.append(data['gene'])
            gene_map[data['gene']] = gene_symbol
    
    # 2. Use lookup_post to get detailed gene information
    gene_info = er.lookup_post(
        ids=gene_ids,
        params={
            'expand': 1  # Get expanded information including transcripts
        }
    )
    
    # 3. Process gene information and collect transcript IDs
    gene_info_list = []
    transcript_ids = []
    
    for gene_id, info in gene_info.items():
        gene_symbol = gene_map.get(gene_id)
        if not gene_symbol:
            continue
            
        # Get canonical transcript if available
        canonical_transcript = None
        if 'Transcript' in info:
            for transcript in info['Transcript']:
                if transcript.get('is_canonical', 0) == 1:
                    canonical_transcript = transcript
                    transcript_ids.append(transcript['id'])
                    break
        
        gene_info_list.append({
            'external_gene_name': gene_symbol,
            'ensembl_gene_id': gene_id,
            'chromosome_name': info.get('seq_region_name'),
            'start_position': info.get('start'),
            'end_position': info.get('end'),
            'strand': info.get('strand'),
            'version': info.get('version'),
            'ensembl_transcript_id': canonical_transcript['id'] if canonical_transcript else None,
            'transcript_version': canonical_transcript.get('version') if canonical_transcript else None,
            'protein_id': canonical_transcript.get('Translation', {}).get('id') if canonical_transcript else None
        })
    
    # Create DataFrame from gene info
    gene_info_df = pd.DataFrame(gene_info_list)
    
    # 4. Get haplotypes using haplosaurus
    if transcript_ids:
        haplotypes = hs.get_haplotypes(
            tx_ids=transcript_ids,
            species=species,
            params=hs.config.PARAMS_HAPLOTYPES,
            use_protein_ids=False,
            verbose=True
        )
        
        # 5. Extract sequences
        sequences_dict, missing_seqs = hs.get_haplotype_seqs(
            haplotypes=haplotypes,
            aligned=1,
            as_msa=False,
            return_missing=True,
            use_protein_ids=False,
            add_haplotype_names=True
        )
    else:
        sequences_dict, missing_seqs = {}, []
    
    return gene_info_df, sequences_dict, missing_seqs

# Example usage:
genes = ["BRCA1", "BRCA2", "TP53"]
gene_info, sequences_dict, missing_seqs = get_gene_sequences(genes)

# Print gene information
print("\nGene Information:")
print(gene_info)

# Print sequence information
if sequences_dict:  # Check if we have any sequences
    print("\nSequence Information:")
    for tx_id, seq_data in sequences_dict.items():
        print(f"\nTranscript: {tx_id}")
        print(f"Number of haplotype sequences: {len(seq_data)}")
        if seq_data:  # If there are any sequences
            first_seq = seq_data[0]  # Get first (name, sequence) tuple
            if isinstance(first_seq, tuple):
                name, seq = first_seq
                print(f"First haplotype name: {name}")
                print(f"First sequence (first 50 aa):")
                print(seq[:50] + "...")

# Print missing sequences
if missing_seqs:
    print("\nTranscripts with no sequences found:")
    for tx_id in missing_seqs:
        print(f"- {tx_id}")

Loading ==> /home/caom/.cache/ensembl_rest/xref_external/cfdb5367288fb30c6907a91c0cd6302a.json.gz


  0%|          | 0/1 [00:00<?, ?it/s]

Getting haplotypes:   0%|          | 0/3 [00:00<?, ?it/s]

Getting haplotype sequences:   0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/3 [00:00<?, ?it/s]


Gene Information:
  external_gene_name ensembl_gene_id chromosome_name  start_position  \
0              BRCA1         LRG_292         LRG_292           92501   
1              BRCA2         LRG_293         LRG_293            5001   
2               TP53         LRG_321         LRG_321            5001   

   end_position  strand  version ensembl_transcript_id  transcript_version  \
0        173689       1        1             LRG_292t1                   1   
1         89193       1        1             LRG_293t1                   1   
2         24149       1        1           LRG_321t1-1                   1   

  protein_id  
0  LRG_292p1  
1  LRG_293p1  
2  LRG_321p1  

Sequence Information:

Transcript: LRG_292t1
Number of haplotype sequences: 1
First haplotype name: LRG
First sequence (first 50 aa):
MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLK...

Transcript: LRG_293t1
Number of haplotype sequences: 1
First haplotype name: LRG
First sequence (first 50 aa):
MPIGSKERPTFFEIFKTR

In [21]:
import src.ensembl_rest as er
import src.haplosaurus as hs
import pandas as pd
import time
import warnings
import ensembl_rest
import gzip
import json
from pathlib import Path

def get_gene_sequences(gene_symbols, species="homo_sapiens", max_retries=3, retry_delay=5):
    """
    Get gene information and their corresponding protein/CDS sequences.
    
    Args:
        gene_symbols (list): List of gene symbols (e.g., ["BRCA1", "BRCA2", "TP53"])
        species (str): Species name (default: "homo_sapiens")
        max_retries (int): Maximum number of retries for API calls
        retry_delay (int): Delay in seconds between retries
        
    Returns:
        tuple: (gene_data, sequences_dict, missing_seqs)
            - gene_data: DataFrame with gene information including all transcripts
            - sequences_dict: Dictionary with protein and CDS sequences
            - missing_seqs: List of transcript IDs with no sequences
    """
    # Initialize Ensembl client
    client = ensembl_rest.EnsemblClient()
    
    # Process each gene symbol
    gene_info_list = []
    transcript_ids = set()  # Use set to avoid duplicates
    
    for gene_symbol in gene_symbols:
        try:
            # Get gene info using symbol_lookup
            gene_info = client.symbol_lookup(species=species, symbol=gene_symbol)
            
            if gene_info:
                # Get canonical transcript ID
                canonical_transcript = gene_info.get('canonical_transcript')
                if canonical_transcript:
                    transcript_id = canonical_transcript.split('.')[0]  # Remove version number
                    gene_info_list.append({
                        'external_gene_name': gene_symbol,
                        'ensembl_gene_id': gene_info['id'],
                        'ensembl_transcript_id': transcript_id,
                        'chromosome_name': gene_info.get('seq_region_name'),
                        'start_position': gene_info.get('start'),
                        'end_position': gene_info.get('end'),
                        'strand': gene_info.get('strand'),
                        'version': gene_info.get('version'),
                        'biotype': gene_info.get('biotype')
                    })
                    transcript_ids.add(transcript_id)
        except Exception as e:
            warnings.warn(f"Error getting info for gene {gene_symbol}: {str(e)}")
    
    # Create DataFrame from gene info
    gene_info_df = pd.DataFrame(gene_info_list)
    
    # 2. Get haplotypes for all transcripts with retry logic
    sequences_dict = {}
    missing_seqs = []
    
    if transcript_ids:
        transcript_ids = list(transcript_ids)  # Convert set to list
        haplotypes = hs.get_haplotypes(
            tx_ids=transcript_ids,
            species=species,
            params=hs.config.PARAMS_HAPLOTYPES,
            use_protein_ids=False,
            verbose=True,
            force=False,  # Use cached data if available
            cache_only=True  # Only use cached data
        )
        
        # Process each transcript's sequences
        for tx_id, haplotype_data in haplotypes.items():
            sequences_dict[tx_id] = {
                'protein': [],
                'protein_ids': {'ref': None, 'hap1': None},  # Store reference and first haplotype IDs
                'cds': []
            }
            
            # Get protein sequences and IDs
            if 'protein_haplotypes' in haplotype_data:
                for i, hap in enumerate(haplotype_data['protein_haplotypes']):
                    name = hap.get('name', 'unknown')
                    protein_seq = hap.get('seq')
                    
                    # Extract protein ID from name (format is usually ENSP00000350283:REF)
                    protein_id = name.split(':')[0] if ':' in name else None
                    
                    if protein_seq:
                        sequences_dict[tx_id]['protein'].append((name, protein_seq))
                        
                        # Store reference and first haplotype IDs
                        if ':REF' in name:
                            sequences_dict[tx_id]['protein_ids']['ref'] = protein_id
                        elif i == 1:  # First non-reference haplotype
                            sequences_dict[tx_id]['protein_ids']['hap1'] = protein_id
            
            # Get CDS sequences
            if 'cds_haplotypes' in haplotype_data:
                for hap in haplotype_data['cds_haplotypes']:
                    name = hap.get('name', 'unknown')
                    cds_seq = hap.get('seq')
                    if cds_seq:
                        sequences_dict[tx_id]['cds'].append((name, cds_seq))
            
            # If no sequences found, add to missing
            if not sequences_dict[tx_id]['protein'] and not sequences_dict[tx_id]['cds']:
                missing_seqs.append(tx_id)
                del sequences_dict[tx_id]
    
    return gene_info_df, sequences_dict, missing_seqs

# Example usage:
if __name__ == "__main__":
    genes = ["BRCA1", "BRCA2", "TP53"]
    gene_info, sequences_dict, missing_seqs = get_gene_sequences(genes)

    # Print gene information
    print("\nGene Information:")
    print(gene_info.to_string())

    # Print sequence information for all genes
    if sequences_dict:
        for gene_symbol in genes:
            # Find the transcript ID for this gene from gene_info
            gene_rows = gene_info[gene_info['external_gene_name'] == gene_symbol]
            if not gene_rows.empty:
                gene_row = gene_rows.iloc[0]
                tx_id = gene_row['ensembl_transcript_id']
                
                if tx_id in sequences_dict:
                    print(f"\n{'='*80}")
                    print(f"Gene: {gene_symbol}")
                    print(f"Transcript ID: {tx_id}")
                    print(f"Protein IDs:")
                    print(f"  Reference: {sequences_dict[tx_id]['protein_ids']['ref']}")
                    print(f"  Haplotype 1: {sequences_dict[tx_id]['protein_ids']['hap1']}")
                    
                    seq_data = sequences_dict[tx_id]
                    
                    # Print only reference and first haplotype sequences
                    if seq_data['protein']:
                        print("\nProtein sequences:")
                        for name, seq in seq_data['protein'][:2]:  # Only first two sequences
                            print(f"\nHaplotype name: {name}")
                            print(f"Protein sequence (first 50 aa):")
                            print(seq[:50] + "...")
                    
                    # Print only reference and first haplotype CDS
                    if seq_data['cds']:
                        print("\nCDS sequences:")
                        for name, seq in seq_data['cds'][:2]:  # Only first two sequences
                            print(f"\nHaplotype name: {name}")
                            print(f"CDS sequence (first 50 nt):")
                            print(seq[:50] + "...")
                else:
                    print(f"\nNo sequences found for {gene_symbol} (transcript {tx_id})")
            else:
                print(f"\nNo information found for gene {gene_symbol}")

    # Print missing sequences
    if missing_seqs:
        print("\nTranscripts with no sequences found:")
        for tx_id in missing_seqs:
            print(f"- {tx_id}")


Getting haplotypes:   0%|          | 0/3 [00:00<?, ?it/s]


Gene Information:
  external_gene_name  ensembl_gene_id ensembl_transcript_id chromosome_name  start_position  end_position  strand  version         biotype
0              BRCA1  ENSG00000012048       ENST00000357654              17        43044295      43170245      -1       26  protein_coding
1              BRCA2  ENSG00000139618       ENST00000380152              13        32315086      32400268       1       19  protein_coding
2               TP53  ENSG00000141510       ENST00000269305              17         7661779       7687546      -1       19  protein_coding

Gene: BRCA1
Transcript ID: ENST00000357654
Protein IDs:
  Reference: ENSP00000350283
  Haplotype 1: ENSP00000350283

Protein sequences:

Haplotype name: ENSP00000350283:REF
Protein sequence (first 50 aa):
MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLK...

Haplotype name: ENSP00000350283:871P>L,1038E>G,1183K>R,1613S>G
Protein sequence (first 50 aa):
MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLK...

CDS sequences:

In [7]:
from src.get_gene_sequences import get_gene_sequences
import pandas as pd
from pathlib import Path

def format_for_alphafold(sequence_data, gene_symbol, output_dir="alphafold_input"):
    """Format protein sequences for AlphaFold prediction."""
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    fasta_files = {}
    
    # Process each sequence
    for name, sequence in sequence_data:
        # Clean haplotype name for filename
        clean_name = name.replace(':', '_').replace('>', '_').replace(',', '_')
        filename = f"{gene_symbol}_{clean_name}.fasta"
        filepath = output_path / filename
        
        # Write FASTA file
        with open(filepath, 'w') as f:
            f.write(f">{name} {gene_symbol}\n")
            # Write sequence in chunks of 80 characters
            for i in range(0, len(sequence), 80):
                f.write(sequence[i:i+80] + '\n')
        
        fasta_files[name] = str(filepath)
    
    return fasta_files

def main():
    # Get BRCA1 sequences
    gene_info, sequences_dict, _ = get_gene_sequences(["BRCA1"])
    
    if not gene_info.empty:
        tx_id = gene_info.iloc[0]['ensembl_transcript_id']
        if tx_id in sequences_dict:
            print(f"\n{'='*80}")
            print("BRCA1 Sequences for AlphaFold Prediction")
            print(f"{'='*80}")
            
            protein_sequences = sequences_dict[tx_id]['protein']
            
            # Format all sequences for AlphaFold
            fasta_files = format_for_alphafold(protein_sequences, "BRCA1")
            
            print(f"\nTotal variants found: {len(protein_sequences)}")
            print("\nSequence Details:")
            
            for hap_name, filepath in fasta_files.items():
                print(f"\n{'-'*40}")
                print(f"Haplotype: {hap_name}")
                print(f"FASTA file: {filepath}")
                
                # Find and print sequence details
                for name, seq in protein_sequences:
                    if name == hap_name:
                        print(f"Sequence length: {len(seq)} amino acids")
                        print("First 50 aa:", seq[:50] + "...")
                        break

if __name__ == "__main__":
    main()


Getting haplotypes:   0%|          | 0/1 [00:00<?, ?it/s]


BRCA1 Sequences for AlphaFold Prediction

Total variants found: 109

Sequence Details:

----------------------------------------
Haplotype: ENSP00000350283:REF
FASTA file: alphafold_input/BRCA1_ENSP00000350283_REF.fasta
Sequence length: 1864 amino acids
First 50 aa: MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLK...

----------------------------------------
Haplotype: ENSP00000350283:871P>L,1038E>G,1183K>R,1613S>G
FASTA file: alphafold_input/BRCA1_ENSP00000350283_871P_L_1038E_G_1183K_R_1613S_G.fasta
Sequence length: 1864 amino acids
First 50 aa: MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLK...

----------------------------------------
Haplotype: ENSP00000350283:871P>L
FASTA file: alphafold_input/BRCA1_ENSP00000350283_871P_L.fasta
Sequence length: 1864 amino acids
First 50 aa: MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLK...

----------------------------------------
Haplotype: ENSP00000350283:693D>N,871P>L,1038E>G,1183K>R,1613S>G
FASTA file: alphafold_input/BRCA1_ENSP00000